# 🛡️ Glu-Stock: 00a_MODEL_RETRAINING_RF
**Phase**: Dynamic Cross-Sectional Intelligence (RF Brain)

This notebook trains the Random Forest 'Super Brain' on a dynamic panel dataset containing 5 years of historical data from the latest active LQ45 constituents. It scrapes the current LQ45 members to ensure the model stays relevant to the most liquid assets.

In [ ]:
# 📦 SECTION 1: INSTALLATION
!pip install -q yfinance firebase-admin pandas scikit-learn joblib lxml html5lib

In [ ]:
# 🏗️ SECTION 2: INFRASTRUCTURE & DYNAMIC DATA
import json, os, firebase_admin, joblib, numpy as np, pandas as pd, yfinance as yf
from firebase_admin import credentials, db
from datetime import datetime
from kaggle_secrets import UserSecretsClient

class KaggleInfra:
    @staticmethod
    def load_secrets():
        user_secrets = UserSecretsClient()
        return {
            "url": user_secrets.get_secret("FIREBASE_URL"),
            "key": json.loads(user_secrets.get_secret("FIREBASE_KEY_JSON"))
        }

class FirebaseHandler:
    def __init__(self, secrets):
        if not firebase_admin._apps:
            cred = credentials.Certificate(secrets['key'])
            firebase_admin.initialize_app(cred, {'databaseURL': secrets['url']})
        self.root_ref = db.reference("glu_stock")
        
    def log_event(self, phase, details):
        self.root_ref.child("history").push({'timestamp': datetime.now().isoformat(), 'phase': phase.upper(), 'details': details})

def get_dynamic_lq45():
    print("🌐 Fetching latest LQ45 constituents...")
    fallback = ["ACES.JK", "ADRO.JK", "AKRA.JK", "AMMN.JK", "AMRT.JK", "ANTM.JK", "ARTO.JK", "ASII.JK", "BBCA.JK", "BBNI.JK", "BBRI.JK", "BBTN.JK", "BMRI.JK", "BRIS.JK", "BRPT.JK", "BUKA.JK", "CPIN.JK", "CTRA.JK", "ESSA.JK", "EXCL.JK", "GGRM.JK", "GOTO.JK", "HRUM.JK", "ICBP.JK", "INCO.JK", "INDF.JK", "INKP.JK", "INTP.JK", "ISAT.JK", "ITMG.JK", "KLBF.JK", "MAPI.JK", "MBMA.JK", "MDKA.JK", "MEDC.JK", "MTEL.JK", "PGAS.JK", "PGEO.JK", "PTBA.JK", "SIDO.JK", "SMGR.JK", "SRTG.JK", "TLKM.JK", "TPIA.JK", "UNTR.JK"]
    try:
        tables = pd.read_html('https://id.wikipedia.org/wiki/LQ45')
        for df in tables:
            if 'Kode' in df.columns:
                tickers = (df['Kode'] + '.JK').tolist()
                print("✅ Dynamically fetched LQ45 list.")
                return tickers
    except Exception as e:
        print("⚠️ Wikipedia fetch failed. Using fallback LQ45 list.")
    return fallback

In [ ]:
# 🧠 SECTION 3: CORE LOGIC (Panel Training Pipeline)
from sklearn.ensemble import RandomForestClassifier

def build_panel_data(universe, period="5y"):
    all_X, all_y = [], []
    print(f"📉 Fetching {period} of data for {len(universe)} tickers...")
    for ticker in universe:
        df = yf.download(ticker, period=period, progress=False)
        if len(df) > 100:
            X = df[['Close']].pct_change().dropna().values.reshape(-1, 1)
            y = (df['Close'].shift(-1) > df['Close']).iloc[:-1].values.astype(int)
            all_X.append(X[:len(y)])
            all_y.append(y)
    print(f"✅ Successfully aggregated market data.")
    return np.vstack(all_X), np.concatenate(all_y)

def train_rf(X, y):
    print(f"🧠 Training Super RF Brain on {len(y)} samples...")
    model = RandomForestClassifier(n_estimators=150, max_depth=10, min_samples_split=5)
    model.fit(X, y)
    return model

In [ ]:
# 🚀 SECTION 4: MAIN EXECUTION
def run_retrain():
    secrets = KaggleInfra.load_secrets()
    fb = FirebaseHandler(secrets)
    output_dir = "/kaggle/working/"
    
    universe = get_dynamic_lq45()
    
    # 1. Build Panel & Train RF
    X_train, y_train = build_panel_data(universe)
    rf_model = train_rf(X_train, y_train)
    accuracy = rf_model.score(X_train, y_train)
    
    brain_data = {"model": rf_model, "features": ["Close"], "accuracy": accuracy, "trained_at": datetime.now().isoformat()}
    joblib.dump(brain_data, os.path.join(output_dir, "glu_brain_v1.joblib"))
    
    fb.log_event("RETRAINING_RF", f"Completed RF panel update on {len(universe)} dynamic LQ45 tickers (Acc: {accuracy:.2f}).")
    print(f"✅ RF Model updated successfully in WORKING directory with {len(y_train)} experiences.")

run_retrain()